# Notebook 16 — CGCS-Constrained Adaptive Control

This notebook consolidates the control-stack pattern from Notebooks 10–15 into one controller test:

- robust innovation gating
- adaptive gate widening under disturbance
- CGCS phase-lock monitoring
- threshold-margin scoring
- recovery / re-entry diagnostics
- clean export of figures, results, docs, and zip output

Repository target:

`github.com/thinkthoughts/quantum-hardware-company/control-stack`

In [ ]:
import os, json, math, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "16"
SLUG = "cgcs_constrained_adaptive_control"
TITLE = "CGCS-Constrained Adaptive Control"

FIG_DIR = Path("figures") / SLUG
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")
OUTPUT_DIR = Path("outputs")

for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(9423)

THRESHOLD = 1 / np.sqrt(1**2 + 1**2)
print(f"CGCS 45° threshold = {THRESHOLD:.4f}")

## Synthetic calibration stream

In [ ]:
N = 84
blocks = np.arange(N)

# disturbance windows
noise_burst = (blocks >= 28) & (blocks <= 40)
model_mismatch = (blocks >= 53) & (blocks <= 66)

omega_true = (
    0.045*np.sin(0.34*blocks + 0.2)
    + 0.025*np.sin(0.77*blocks - 0.4)
    + 0.010*np.sin(0.13*blocks)
)
b_true = (
    0.003 + 0.00075*blocks
    + 0.006*np.sin(0.22*blocks)
    + 0.004*np.sin(0.48*blocks + 0.7)
)

omega_noise = rng.normal(0, 0.0035, N)
b_noise = rng.normal(0, 0.0025, N)

omega_noise[noise_burst] += rng.normal(0, 0.020, noise_burst.sum())
b_noise[noise_burst] += rng.normal(0, 0.018, noise_burst.sum())

omega_meas = omega_true + omega_noise
b_meas = b_true + b_noise

# model mismatch: biased command demand / drift process jump
omega_meas[model_mismatch] += 0.035*np.sin(0.85*blocks[model_mismatch]) + 0.020
b_meas[model_mismatch] += 0.035 + 0.010*np.sin(0.55*blocks[model_mismatch])

# isolated outliers
for idx, amp in [(17, 0.18), (46, -0.20), (72, 0.15)]:
    omega_meas[idx] += amp
for idx, amp in [(24, 0.12), (69, -0.14)]:
    b_meas[idx] += amp

target_phase = np.column_stack([omega_true, b_true])
measured_phase = np.column_stack([omega_meas, b_meas])

pd.DataFrame({
    "block": blocks,
    "omega_true": omega_true,
    "b_true": b_true,
    "omega_measured": omega_meas,
    "b_measured": b_meas,
    "noise_burst": noise_burst,
    "model_mismatch": model_mismatch,
}).head()

## Helpers

In [ ]:
def moving_average(x, w=5):
    y = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i-w+1)
        y[i] = np.mean(x[lo:i+1])
    return y

def scalar_kalman(z, q=8e-4, r=2e-4):
    x = np.zeros_like(z, dtype=float)
    p = 1.0
    x[0] = z[0]
    for i in range(1, len(z)):
        p = p + q
        k = p / (p + r)
        x[i] = x[i-1] + k*(z[i] - x[i-1])
        p = (1-k)*p
    return x

def robust_scalar_kalman(z, q=8e-4, r=2e-4, gate=0.075):
    x = np.zeros_like(z, dtype=float)
    rejected = np.zeros(len(z), dtype=bool)
    p = 1.0
    x[0] = z[0]
    for i in range(1, len(z)):
        pred = x[i-1]
        p = p + q
        innovation = z[i] - pred
        if abs(innovation) > gate:
            rejected[i] = True
            x[i] = pred + np.sign(innovation)*gate*0.25
            p *= 1.05
            continue
        k = p / (p + r)
        x[i] = pred + k*innovation
        p = (1-k)*p
    return x, rejected

def adaptive_gate_kalman(z, q=8e-4, r=2e-4, base_gate=0.075, max_gate=0.22, alpha=0.08):
    x = np.zeros_like(z, dtype=float)
    gates = np.zeros(len(z), dtype=float)
    rejected = np.zeros(len(z), dtype=bool)
    p = 1.0
    gate = base_gate
    x[0] = z[0]
    gates[0] = gate
    recent_rejects = 0
    for i in range(1, len(z)):
        pred = x[i-1]
        p = p + q
        innovation = z[i] - pred
        if abs(innovation) > gate:
            rejected[i] = True
            recent_rejects += 1
            gate = min(max_gate, gate*(1+alpha*recent_rejects) + 0.01)
            x[i] = pred + np.sign(innovation)*gate*0.20
            p *= 1.10
        else:
            recent_rejects = max(0, recent_rejects-1)
            gate = max(base_gate, gate*0.985)
            k = p / (p + r)
            x[i] = pred + k*innovation
            p = (1-k)*p
        gates[i] = gate
    return x, gates, rejected

def cgcs_cosine(pred, target):
    # convert state vector to response-like vector; cosine with offset baseline for stability
    a = np.column_stack([np.ones(len(pred)), pred[:,0], pred[:,1]])
    b = np.column_stack([np.ones(len(target)), target[:,0], target[:,1]])
    dots = np.sum(a*b, axis=1)
    denom = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    return dots / denom

def response_rmse(pred, target):
    # response surrogate: pulse probability mismatch under phase and offset perturbations
    t = np.linspace(0, 10, 240)
    rmses = []
    for (om, bb), (ot, bt) in zip(pred, target):
        y = np.sin((1.0 + om)*t + bb)**2
        y0 = np.sin((1.0 + ot)*t + bt)**2
        rmses.append(np.sqrt(np.mean((y-y0)**2)))
    return np.array(rmses)

def settle_blocks(cosine, threshold=THRESHOLD, settle=0.98):
    below = cosine < threshold
    if not np.any(below):
        return 0
    last = np.where(below)[0].max()
    after = np.where(cosine[last:] >= settle)[0]
    if len(after) == 0:
        return len(cosine) - last
    return int(after[0])

def phase_error_integral(cosine):
    return float(np.sum(np.maximum(0, 1.0 - cosine)))

def save_fig(name):
    path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png"
    plt.savefig(path, dpi=220, bbox_inches="tight")
    print(f"[saved] {path}")
    return path

## Policy construction

In [ ]:
ma = np.column_stack([moving_average(omega_meas, 5), moving_average(b_meas, 5)])

scalar = np.column_stack([
    scalar_kalman(omega_meas, q=8e-4, r=2e-4),
    scalar_kalman(b_meas, q=2e-4, r=1e-4),
])

joint = np.column_stack([
    scalar_kalman(omega_meas, q=1.0e-3, r=1.5e-4),
    scalar_kalman(b_meas, q=3.0e-4, r=8e-5),
])

rob_om, rej_om = robust_scalar_kalman(omega_meas, q=1.0e-3, r=1.5e-4, gate=0.075)
rob_b, rej_b = robust_scalar_kalman(b_meas, q=3.0e-4, r=8e-5, gate=0.060)
robust = np.column_stack([rob_om, rob_b])

ad_om, gate_om, rej_ad_om = adaptive_gate_kalman(omega_meas, q=1.0e-3, r=1.5e-4, base_gate=0.070, max_gate=0.22)
ad_b, gate_b, rej_ad_b = adaptive_gate_kalman(b_meas, q=3.0e-4, r=8e-5, base_gate=0.055, max_gate=0.20)
adaptive_gate = np.column_stack([ad_om, ad_b])

# CGCS constrained adaptive controller: blends robust estimate with adaptive gate estimate,
# then clips response jumps to preserve phase-lock continuity.
cgcs_adaptive = np.zeros_like(adaptive_gate)
cgcs_adaptive[0] = adaptive_gate[0]
for i in range(1, N):
    candidate = 0.72*adaptive_gate[i] + 0.28*robust[i]
    delta = candidate - cgcs_adaptive[i-1]
    max_delta = np.array([0.050, 0.045])
    delta = np.clip(delta, -max_delta, max_delta)
    cgcs_adaptive[i] = cgcs_adaptive[i-1] + delta

# command / constrained MPC-lite: smoother and more conservative under mismatch
constrained = np.zeros_like(joint)
constrained[0] = joint[0]
for i in range(1, N):
    gain = 0.65 if not model_mismatch[i] else 0.38
    proposed = joint[i]
    constrained[i] = constrained[i-1] + gain*(proposed - constrained[i-1])
    constrained[i] = np.clip(constrained[i], [-0.25, -0.05], [0.25, 0.12])

policies = {
    "oracle": target_phase,
    "none": np.zeros_like(target_phase),
    "moving_average": ma,
    "scalar_kalman": scalar,
    "joint_kalman": joint,
    "robust_gated_kalman": robust,
    "adaptive_gate_kalman": adaptive_gate,
    "cgcs_adaptive_control": cgcs_adaptive,
    "constrained_mpc": constrained,
}

summary_rows = []
cosines = {}
rmses = {}
for name, pred in policies.items():
    c = cgcs_cosine(pred, target_phase)
    r = response_rmse(pred, target_phase)
    cosines[name] = c
    rmses[name] = r
    summary_rows.append({
        "policy": name,
        "mean_rmse": float(np.mean(r)),
        "max_rmse": float(np.max(r)),
        "min_cosine": float(np.min(c)),
        "mean_cosine": float(np.mean(c)),
        "blocks_below_threshold": int(np.sum(c < THRESHOLD)),
        "settle_blocks": settle_blocks(c),
        "phase_error_integral": phase_error_integral(c),
    })

summary = pd.DataFrame(summary_rows).sort_values(["mean_rmse", "blocks_below_threshold"])
summary

## Figures

In [ ]:
plt.figure(figsize=(14, 6))
plt.axhline(THRESHOLD, linestyle="--", label="45° threshold")
plt.axvspan(28, 40, alpha=0.18, label="noise burst")
plt.axvspan(53, 66, alpha=0.12, label="model mismatch")
for name in ["none", "moving_average", "joint_kalman", "robust_gated_kalman", "adaptive_gate_kalman", "cgcs_adaptive_control", "constrained_mpc", "oracle"]:
    plt.plot(blocks, cosines[name], marker="o", linewidth=1.5, label=name)
plt.title("CGCS-constrained adaptive control: phase-lock stability")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.ylim(0.55, 1.02)
plt.legend(ncol=2, fontsize=9)
save_fig("cgcs_stability_comparison")
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
plt.axvspan(28, 40, alpha=0.18, label="noise burst")
plt.axvspan(53, 66, alpha=0.12, label="model mismatch")
for name in ["none", "moving_average", "joint_kalman", "robust_gated_kalman", "adaptive_gate_kalman", "cgcs_adaptive_control", "constrained_mpc", "oracle"]:
    plt.plot(blocks, rmses[name], marker="o", linewidth=1.4, label=name)
plt.title("CGCS-constrained adaptive control: response RMSE comparison")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target response")
plt.legend(ncol=2, fontsize=9)
save_fig("response_rmse_comparison")
plt.show()

In [ ]:
ranked = summary.sort_values("mean_rmse")
x = np.arange(len(ranked))
plt.figure(figsize=(13, 5))
plt.bar(x - 0.18, ranked["mean_rmse"], width=0.36, label="mean RMSE")
plt.bar(x + 0.18, ranked["max_rmse"], width=0.36, label="max RMSE")
plt.xticks(x, ranked["policy"], rotation=25, ha="right")
plt.title("CGCS-constrained adaptive control: policy ranking")
plt.ylabel("response RMSE")
plt.legend()
save_fig("policy_ranking_summary")
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.axvspan(28, 40, alpha=0.18, label="noise burst")
plt.axvspan(53, 66, alpha=0.12, label="model mismatch")
plt.plot(blocks, omega_true, linewidth=2.0, label="true Ω drift")
plt.plot(blocks, omega_meas, marker="o", alpha=0.65, label="measured Ω")
for name in ["joint_kalman", "robust_gated_kalman", "adaptive_gate_kalman", "cgcs_adaptive_control", "constrained_mpc"]:
    plt.plot(blocks, policies[name][:,0], linewidth=1.8, label=name)
plt.title("CGCS-constrained adaptive control: Ω tracking")
plt.xlabel("calibration block")
plt.ylabel("Ω drift estimate / command")
plt.legend(ncol=2, fontsize=9)
save_fig("omega_tracking")
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.axvspan(28, 40, alpha=0.18, label="noise burst")
plt.axvspan(53, 66, alpha=0.12, label="model mismatch")
plt.plot(blocks, b_true, linewidth=2.0, label="true B drift")
plt.plot(blocks, b_meas, marker="o", alpha=0.65, label="measured B")
for name in ["joint_kalman", "robust_gated_kalman", "adaptive_gate_kalman", "cgcs_adaptive_control", "constrained_mpc"]:
    plt.plot(blocks, policies[name][:,1], linewidth=1.8, label=name)
plt.title("CGCS-constrained adaptive control: B tracking")
plt.xlabel("calibration block")
plt.ylabel("B drift estimate / command")
plt.legend(ncol=2, fontsize=9)
save_fig("offset_tracking")
plt.show()

In [ ]:
plt.figure(figsize=(14, 4.5))
plt.axvspan(28, 40, alpha=0.18, label="noise burst")
plt.axvspan(53, 66, alpha=0.12, label="model mismatch")
plt.plot(blocks, gate_om, label="adaptive Ω gate")
plt.plot(blocks, gate_b, label="adaptive B gate")
plt.axhline(0.075, linestyle="--", label="fixed robust Ω gate")
plt.title("CGCS-constrained adaptive control: adaptive gate trace")
plt.xlabel("calibration block")
plt.ylabel("innovation gate")
plt.legend()
save_fig("adaptive_gate_trace")
plt.show()

In [ ]:
plt.figure(figsize=(13, 4.5))
for y, (name, rej) in enumerate([
    ("robust_gated_kalman", rej_om | rej_b),
    ("adaptive_gate_kalman", rej_ad_om | rej_ad_b),
]):
    xs = blocks[rej]
    plt.scatter(xs, np.full_like(xs, y), marker="x", s=70, label=name)
plt.yticks([0,1], ["robust_gated_kalman", "adaptive_gate_kalman"])
plt.title("CGCS-constrained adaptive control: rejection events")
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.legend()
save_fig("rejection_events")
plt.show()

In [ ]:
metrics = summary.set_index("policy").loc[
    ["oracle", "cgcs_adaptive_control", "adaptive_gate_kalman", "robust_gated_kalman", "joint_kalman", "constrained_mpc", "moving_average", "none"]
].reset_index()

plt.figure(figsize=(13, 5))
x = np.arange(len(metrics))
plt.bar(x, metrics["phase_error_integral"])
plt.xticks(x, metrics["policy"], rotation=25, ha="right")
plt.title("CGCS-constrained adaptive control: phase-error integral")
plt.ylabel("Σ max(0, 1 - cosine)")
save_fig("phase_error_integral")
plt.show()

In [ ]:
plt.figure(figsize=(13, 5))
x = np.arange(len(metrics))
plt.bar(x, metrics["blocks_below_threshold"])
plt.xticks(x, metrics["policy"], rotation=25, ha="right")
plt.title("CGCS-constrained adaptive control: blocks below 45° threshold")
plt.ylabel("blocks below threshold")
save_fig("blocks_below_threshold")
plt.show()

In [ ]:
# Worst-case block for selected final controller
controller = "cgcs_adaptive_control"
worst_block = int(np.argmax(rmses[controller]))
t = np.linspace(0, 10, 300)

plt.figure(figsize=(13, 5))
for name in ["oracle", "none", "moving_average", "joint_kalman", "robust_gated_kalman", "adaptive_gate_kalman", "cgcs_adaptive_control", "constrained_mpc"]:
    om, bb = policies[name][worst_block]
    y = np.sin((1+om)*t + bb)**2
    plt.plot(t, y, linewidth=1.8, label=name)
plt.title(f"CGCS-constrained adaptive control: worst-case block {worst_block}")
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.legend(ncol=2, fontsize=9)
save_fig("worst_case_block_comparison")
plt.show()

## Save results, docs, and zip

In [ ]:
# Save CSV/JSON results
summary_path = RESULTS_DIR / f"{SLUG}_summary.csv"
json_path = RESULTS_DIR / f"{SLUG}_summary.json"
summary.to_csv(summary_path, index=False)
with open(json_path, "w") as f:
    json.dump({
        "notebook": NOTEBOOK_ID,
        "slug": SLUG,
        "threshold": float(THRESHOLD),
        "best_policy_mean_rmse": str(summary.iloc[0]["policy"]),
        "worst_block_cgcs_adaptive_control": worst_block,
        "summary": summary.to_dict(orient="records"),
    }, f, indent=2)

print(f"[saved] {summary_path}")
print(f"[saved] {json_path}")

# Markdown docs summary with repo-relative image paths
figure_names = [
    ("CGCS stability comparison", "cgcs_stability_comparison"),
    ("Response RMSE comparison", "response_rmse_comparison"),
    ("Policy ranking summary", "policy_ranking_summary"),
    ("Ω tracking", "omega_tracking"),
    ("Offset tracking", "offset_tracking"),
    ("Adaptive gate trace", "adaptive_gate_trace"),
    ("Rejection events", "rejection_events"),
    ("Phase-error integral", "phase_error_integral"),
    ("Blocks below threshold", "blocks_below_threshold"),
    ("Worst-case block comparison", "worst_case_block_comparison"),
]

figure_sections = []
for title, name in figure_names:
    figure_sections.append(
        f"### {title}\n\n"
        f"![{title}](../figures/{SLUG}/{NOTEBOOK_ID}_{SLUG}_{name}.png)\n"
    )

top = summary.sort_values("mean_rmse").head(5)
ranking_table = top[["policy", "mean_rmse", "max_rmse", "min_cosine", "blocks_below_threshold"]].to_markdown(index=False)

md_text = f'''# {TITLE} (Notebook {NOTEBOOK_ID})

This notebook consolidates adaptive Kalman filtering, robust innovation gating, CGCS phase-lock monitoring, and constrained control into a single controller comparison.

## Configuration

- Calibration blocks: {N}
- CGCS 45° threshold: {THRESHOLD:.4f}
- Noise burst window: blocks 28–40
- Model mismatch window: blocks 53–66
- Output figure folder: `figures/{SLUG}/`
- Results files:
  - `results/{SLUG}_summary.csv`
  - `results/{SLUG}_summary.json`

## Top policies by mean RMSE

{ranking_table}

## Key observations

- `cgcs_adaptive_control` combines adaptive innovation gates with CGCS continuity clipping.
- Robust gating reduces outlier damage but may under-react after repeated disturbance.
- Adaptive gates widen during unstable periods and contract after measurements stabilize.
- CGCS threshold tracking separates raw response error from phase-lock failure.
- Worst-case diagnostics identify where response correction still lags.

## Figures

{chr(10).join(figure_sections)}

## Next direction

Notebook 17 can turn this controller into a closed-loop benchmark suite:

- repeated Monte Carlo seeds
- disturbance families
- controller ablation grid
- pass/fail envelopes
- paper-ready aggregate tables
'''

docs_path = DOCS_DIR / f"{SLUG}.md"
with open(docs_path, "w") as f:
    f.write(md_text)

print(f"[saved] {docs_path}")

# Zip all output artifacts
zip_name = f"{NOTEBOOK_ID}_{SLUG}_outputs.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        for path in folder.rglob("*"):
            if path.is_file() and (SLUG in str(path) or path.parent == FIG_DIR):
                z.write(path, arcname=str(path))
print(f"[saved] {zip_name}")

In [ ]:
# Colab download helper
try:
    from google.colab import files
    files.download(f"{NOTEBOOK_ID}_{SLUG}_outputs.zip")
except Exception as e:
    print("Download helper skipped or unavailable.")
    print("Zip file is available at:", f"{NOTEBOOK_ID}_{SLUG}_outputs.zip")
    print("Reason:", repr(e))